# Gemini 3 Batch Transcription Pipeline

This Colab orchestrates the transcription of audio segments using Google's **Gemini 3** model.

### Workflow Overview:
1.  **Job Submission**: Reads a JSONL manifest from GCS and submits asynchronous batch jobs in chunks of 15 segments. It automatically checks for existing transcripts and skips already processed files to avoid duplicate work and optimize API costs (Delta processing).
2.  **Organization**: Results are written directly to GCS (`gs://wd-transcription-data/transcripts/one_hour_pilot_audio/gemini_3_1_pro_preview`).

In [ ]:
!pip install loguru

In [ ]:
import json
import re
import sys
import time

from google import genai
from google.cloud import storage
from google.colab import auth
from loguru import logger

In [ ]:
# @title Define constants and initial logging

MODEL_ID = "gemini-3.1-pro-preview"
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "transcription": {"type": "string"}
    },
    "required": ["transcription"]
}

GCP_PROJECT_ID = "" # @param {type:"string"}
GCS_BUCKET="" # @param {type:"string"}
GCS_INPUT_DIR="segmented_audio/one_hour_pilot_audio"

# Pipeline Control
OVERWRITE_EXISTING = True # @param {type:"boolean"}

# Create a model-specific directory name
MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)
GCS_OUTPUT_BASE = f"transcripts/one_hour_pilot_audio/{MODEL_ID_DIR}"
GCP_LOCATION="global"

# Segmentation manifest path
MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/batch_manifest.jsonl"

BATCH_INPUT_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/vertex_batch_input.jsonl"
BATCH_OUTPUT_ROOT = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/batch_results/"

# The consistent final path for consolidated results
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"

SYSTEM_PROMPT = """
You are an expert transcriptionist specializing in emergency dispatch radio communications.
Your goal is to provide the most accurate verbatim transcript possible.

CONTEXT & TERMINOLOGY:
- Listen for Unit IDs/Call Signs (e.g., 'Engine 6333', 'Battalion 6', 'Medic 1422', '1 David 4').
- Listen for Dispatchers (often called 'Dispatch', 'Rivercom', 'County', 'Central', or 'Alarm').
- Use digits for numbers and codes (e.g., 'Code 4', '10-24', '10:36').
- Recognize terminology like 'quarters' (returning to station), 'MVC' (motor vehicle crash), and 'en route'.

CRITICAL RULES:
1. ONLY provide the transcription text with no newlines.
2. DO NOT GUESS. If a word, number, or phrase is unintelligible due to noise, output [UNINTELLIGIBLE] rather than hallucinating.
3. When transcribing numbers, write the digits (e.g., 1.7, 100). Never invent numbers.
4. Use standard punctuation and capitalization.
5. Do not add metadata, speaker labels, or noise descriptions.
6. Maintain professional codes exactly as spoken.
"""

GENERATION_CONFIG = {
    "system_instruction": SYSTEM_PROMPT,
    "temperature": 0.0,          # Set to 0.0 for maximum factual determinism
    "top_p": 0.95,               # Nucleus sampling
    "top_k": 40,                 # Top-K sampling
    "frequency_penalty": 0.0,    # Removed penalty to prevent it from avoiding correct but repeated words
    "presence_penalty": 0.0,     # No penalty for staying on topic
    "response_mime_type": "application/json",
    "response_schema": RESPONSE_SCHEMA,
    # Increased thinking budget to help the model process noisy audio more deeply
    "thinkingConfig": {"thinkingBudget": 4096}
}

logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}")

In [ ]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Allow access to the GCS bucket to the VertexAI Service Agent
!gcloud storage buckets add-iam-policy-binding gs://{GCS_BUCKET} \
    --member="serviceAccount:service-$(gcloud projects describe {GCP_PROJECT_ID} --format='value(projectNumber)')@gcp-sa-aiplatform.iam.gserviceaccount.com" \
    --role="roles/storage.objectViewer"

In [ ]:
def get_failed_uris(results_uri: str, bucket_name: str) -> list[str]:
    """Identifies URIs that resulted in errors in the final output file."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    path = results_uri.replace(f"gs://{bucket_name}/", "")
    blob = storage_client.bucket(bucket_name).blob(path)
    if not blob.exists():
        return []
    lines = blob.download_as_text().strip().split("\n")
    failed = []
    for line in lines:
        if not line.strip():
            continue
        data = json.loads(line)
        if data.get("status"):
            uri = data["request"]["contents"][0]["parts"][0]["file_data"]["file_uri"]
            failed.append(uri)
    return failed

def create_retry_manifest(failed_uris: list[str], original_manifest_uri: str, retry_manifest_uri: str) -> None:
    """Filters the original manifest for only the failed URIs."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket_name = original_manifest_uri.replace("gs://", "").split("/")[0]
    path = "/".join(original_manifest_uri.replace("gs://", "").split("/")[1:])
    content = storage_client.bucket(bucket_name).blob(path).download_as_text().strip().split("\n")
    retry_lines = [line for line in content if line.strip() and json.loads(line)["audio_filepath"] in failed_uris]
    out_bucket = retry_manifest_uri.replace("gs://", "").split("/")[0]
    out_path = "/".join(retry_manifest_uri.replace("gs://", "").split("/")[1:])
    storage_client.bucket(out_bucket).blob(out_path).upload_from_string("\n".join(retry_lines))

def prepare_batch_manifest(input_manifest_uri: str, output_batch_manifest_uri: str, *, overwrite: bool = False) -> str | None:
    """Prepares the JSONL manifest, skipping already processed files if overwrite is False."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    m_bucket = input_manifest_uri.replace("gs://", "").split("/")[0]
    m_path = "/".join(input_manifest_uri.replace("gs://", "").split("/")[1:])
    manifest_content = storage_client.bucket(m_bucket).blob(m_path).download_as_text().strip().split("\n")

    processed_uris = set()
    if not overwrite:
        r_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        r_path = "/".join(CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:])
        results_blob = storage_client.bucket(r_bucket).blob(r_path)
        if results_blob.exists():
            for line in results_blob.download_as_text().strip().split("\n"):
                if line.strip():
                    data = json.loads(line)
                    if not data.get("status"):
                        processed_uris.add(data["request"]["contents"][0]["parts"][0]["file_data"]["file_uri"])

    batch_entries = []
    for line in manifest_content:
        if not line.strip():
            continue
        entry = json.loads(line)
        if entry["audio_filepath"] in processed_uris:
            continue

        batch_entry = {
            "request": {
                "contents": [{
                    "role": "user",
                    "parts": [
                        {"file_data": {"file_uri": entry["audio_filepath"], "mime_type": "audio/flac"}},
                        {"text": "Transcribe this emergency radio communication segment."}
                    ]
                }],
                "generation_config": GENERATION_CONFIG
            }
        }
        batch_entries.append(json.dumps(batch_entry))

    if not batch_entries:
        return None

    out_bucket = output_batch_manifest_uri.replace("gs://", "").split("/")[0]
    out_path = "/".join(output_batch_manifest_uri.replace("gs://", "").split("/")[1:])
    storage_client.bucket(out_bucket).blob(out_path).upload_from_string("\n".join(batch_entries))
    logger.info(f"Prepared {len(batch_entries)} segments for processing at {output_batch_manifest_uri}")
    return output_batch_manifest_uri

def submit_vertex_batch_job(model_id: str, input_uri: str, output_root: str) -> any:
    """Submits a batch prediction job to Vertex AI using the Generative AI SDK."""
    client = genai.Client(vertex=True, project=GCP_PROJECT_ID, location=GCP_LOCATION)
    job = client.batches.create(model=f"publishers/google/models/{model_id}", src=input_uri, dest=output_root)
    logger.info(f"Batch job submitted! ID: {job.name}")
    while True:
        job = client.batches.get(name=job.name)
        state = getattr(job.state, "name", str(job.state))
        if state in ["JOB_STATE_SUCCEEDED", "SUCCEEDED", "JOB_STATE_FAILED", "FAILED", "JOB_STATE_CANCELLED"]:
            break
        logger.info(f"Current job state: {state}... checking again in 60s")
        time.sleep(60)
    logger.info(f"Job finished with state: {state}")
    return job

def run_automated_retry_pipeline() -> None:
    """Orchestrates checking for failures, creating a retry manifest, and merging results."""
    logger.info("Starting automated error check...")
    failed_uris = get_failed_uris(CONSISTENT_OUTPUT_URI, GCS_BUCKET)
    if not failed_uris:
        logger.info("No failed segments detected. Pipeline complete.")
        return

    logger.info(f"Detected {len(failed_uris)} failures. Creating retry manifest...")
    RETRY_MANIFEST = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/automated_retry_manifest.jsonl"
    create_retry_manifest(failed_uris, MANIFEST_URI, RETRY_MANIFEST)

    retry_job = submit_vertex_batch_job(MODEL_ID, RETRY_MANIFEST, BATCH_OUTPUT_ROOT)
    if retry_job and getattr(retry_job.state, "name", str(retry_job.state)) in ["JOB_STATE_SUCCEEDED", "SUCCEEDED"]:
        merge_retry_results(retry_job, GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI)
    validate_transcription_results(MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET)

def consolidate_main_results(job: any, bucket_name: str, output_base: str, target_uri: str) -> None:
    """Robustly finds and moves results from the main batch job."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    prefix = f"{output_base}/batch_results/prediction-model-"
    blobs = list(bucket.list_blobs(prefix=prefix))
    prediction_blobs = sorted([b for b in blobs if "predictions.jsonl" in b.name], key=lambda x: x.updated, reverse=True)
    if prediction_blobs:
        logger.info(f"Consolidating results from {prediction_blobs[0].name} to {target_uri}")
        bucket.copy_blob(prediction_blobs[0], bucket, target_uri.replace(f"gs://{bucket_name}/", ""))

def merge_retry_results(retry_job: any, bucket_name: str, output_base: str, target_uri: str) -> None:
    """Finds the most recent retry output and appends unique successes to the final file."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    prefix = f"{output_base}/batch_results/prediction-model-"
    blobs = sorted([b for b in bucket.list_blobs(prefix=prefix) if "predictions.jsonl" in b.name], key=lambda x: x.updated, reverse=True)
    if len(blobs) < 2:
        return
    retry_blob = blobs[0]
    retry_lines = retry_blob.download_as_text().strip().split("\n")
    new_successes = [line for line in retry_lines if line.strip() and not json.loads(line).get("status")]
    if new_successes:
        target_path = target_uri.replace(f"gs://{bucket_name}/", "")
        target_blob = bucket.blob(target_path)
        existing = target_blob.download_as_text().strip() if target_blob.exists() else ""
        seen = {json.loads(line)["request"]["contents"][0]["parts"][0]["file_data"]["file_uri"] for line in existing.split("\n") if line.strip()}
        to_add = [line for line in new_successes if json.loads(line)["request"]["contents"][0]["parts"][0]["file_data"]["file_uri"] not in seen]
        if to_add:
            target_blob.upload_from_string((existing + "\n" + "\n".join(to_add)).strip())

def validate_transcription_results(manifest_uri: str, results_uri: str, bucket_name: str) -> None:
    """Checks final result count against the original manifest count."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    m_bucket = manifest_uri.replace("gs://", "").split("/")[0]
    m_path = "/".join(manifest_uri.replace("gs://", "").split("/")[1:])
    expected = len([line for line in storage_client.bucket(m_bucket).blob(m_path).download_as_text().strip().split("\n") if line.strip()])
    r_path = results_uri.replace(f"gs://{bucket_name}/", "")
    results_blob = storage_client.bucket(bucket_name).blob(r_path)
    if results_blob.exists():
        found = len([line for line in results_blob.download_as_text().strip().split("\n") if line.strip()])
        logger.info(f"Pipeline Validation: Expected {expected}, Found {found}.")
    else:
        logger.warning(f"Validation failed: Result file {results_uri} not found.")

In [ ]:
# @title Main Job Submission
actual_batch_input = prepare_batch_manifest(
    input_manifest_uri=MANIFEST_URI,
    output_batch_manifest_uri=BATCH_INPUT_URI,
    overwrite=OVERWRITE_EXISTING
)

if actual_batch_input:
    logger.info("Step 2: Submitting main batch job...")
    main_job = submit_vertex_batch_job(
        model_id=MODEL_ID,
        input_uri=actual_batch_input,
        output_root=BATCH_OUTPUT_ROOT
    )
    if main_job:
        state = getattr(main_job.state, "name", str(main_job.state))
        if state in ["JOB_STATE_SUCCEEDED", "SUCCEEDED", "JOB_STATE_PARTIALLY_SUCCEEDED", "PARTIALLY_SUCCEEDED"]:
            logger.info("Main job completed. Consolidating results...")
            consolidate_main_results(main_job, GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI)
            # Run immediate validation after consolidation
            validate_transcription_results(MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET)
        else:
             logger.error(f"Main job failed with state: {state}. Skipping consolidation.")
else:
    logger.info("Skipping main job submission (no new segments to process).")
    validate_transcription_results(MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET)

In [ ]:
# @title Automated Retry & Merge Orchestrator
logger.info("Executing automated retry and merge pipeline...")
run_automated_retry_pipeline()